# StrataForge notebook usage
This notebook shows the real public usage surface for the current codebase:

- deterministic parse
- deterministic tree build
- LLM-assisted tree build
- gateway-backed repair + summarization
- artifact inspection

In [3]:
# Cell 1 — bootstrap notebook imports and repo path

from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from uuid import uuid4
from pprint import pprint

REPO_ROOT = Path("/home/pruthvi/projects/StrataForge").resolve()
SRC_ROOT = REPO_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("REPO_ROOT =", REPO_ROOT)
print("SRC_ROOT  =", SRC_ROOT)

REPO_ROOT = /home/pruthvi/projects/StrataForge
SRC_ROOT  = /home/pruthvi/projects/StrataForge/src


In [4]:
# Cell 2 — import current public entrypoints and contracts

import fitz
import pypdf

from strataforge.constants import (
    EXPECTED_PYMUPDF_VERSION,
    EXPECTED_PYPDF_VERSION,
)
from strataforge import (
    ParseRequest,
    ParserSettings,
    TreeBuildRequest,
    TreeSettings,
    TocDetectionResponse,
)
from strataforge.ingest import parse_document
from strataforge.tree import build_tree
from strataforge.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayRepairEngine,
    GatewayRequest,
    GatewayService,
    LiteLLMProviderConfig,
    OpenAIProviderConfig,
    StructuredOutputMode,
)
from strataforge.llm.prompts import build_toc_detection_messages

print("Imports OK")

Imports OK


In [5]:
# Cell 3 — strict runtime sanity check
# Current parser code validates versions against settings during parse.

assert fitz.VersionBind == EXPECTED_PYMUPDF_VERSION, (
    f"Installed PyMuPDF={fitz.VersionBind}, expected={EXPECTED_PYMUPDF_VERSION}"
)
assert pypdf.__version__ == EXPECTED_PYPDF_VERSION, (
    f"Installed pypdf={pypdf.__version__}, expected={EXPECTED_PYPDF_VERSION}"
)

print("PyMuPDF =", fitz.VersionBind)
print("pypdf   =", pypdf.__version__)
print("Runtime versions are aligned with current StrataForge expectations.")

PyMuPDF = 1.27.2
pypdf   = 6.8.0
Runtime versions are aligned with current StrataForge expectations.


In [7]:
# Cell 4 — configure source PDF and run IDs
# Replace SOURCE_PDF with a real input PDF.

SOURCE_PDF = "/home/pruthvi/projects/StrataForge/903000608.pdf"  # <-- replace this
ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "parse_runs"

assert SOURCE_PDF.exists(), f"Missing PDF: {SOURCE_PDF}"

PARSE_RUN_ID = f"parse-{uuid4().hex[:12]}"
TREE_RUN_ID_DET = f"tree-det-{uuid4().hex[:12]}"
TREE_RUN_ID_LLM = f"tree-llm-{uuid4().hex[:12]}"

# For OCR-heavy scanned PDFs, set this to your real tessdata directory.
# Leave as None for native-text PDFs.
OCR_TESSDATA = None
# Example:
# OCR_TESSDATA = Path("/usr/share/tesseract-ocr/5/tessdata")

print("SOURCE_PDF    =", SOURCE_PDF)
print("ARTIFACT_ROOT =", ARTIFACT_ROOT)
print("PARSE_RUN_ID  =", PARSE_RUN_ID)
print("TREE_RUN_ID_DET =", TREE_RUN_ID_DET)
print("TREE_RUN_ID_LLM =", TREE_RUN_ID_LLM)

AttributeError: 'str' object has no attribute 'exists'